# Compute Sampling Performance - Take a long time to run

In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
from typing import Optional, List, Tuple, Dict


# =========================
# CONFIG
# =========================
BASE_DIR = "al_trajectory_data_all"

# Original datasets (pool) and the column that defines the objective (not used for matching, but useful for sanity print)
ORIGINAL_DATASETS = {
    "matbench_steels (composition)": ("matbench_steels.csv", "yield strength"),
    "P3HT_dataset": ("P3HT_dataset.csv", "Conductivity (measured) (S/cm)"),
    "Perovskite_dataset": ("Perovskite_dataset.csv", "Instability index"),
    "Membrane_dataset": ("Membrane_dataset.csv", "Elastic Modulus_mean"),
}

# Output root folder
OUT_ROOT = "Sampling Performance Check"

# Which LLM files to include per dataset folder
LLM_GLOB = "llm_experiment_*_trajectory_*.csv"

# Exclude files you may have generated later
EXCLUDE_SUFFIXES = (
    "_with_rerank_score.csv",
)

# Remove the LAST point from each trajectory (final best point)
DROP_LAST_POINT = True

# For string-based matching fallback (if Experiment Index missing)
# We'll try these columns in the *original dataset* as keys.
ORIG_KEY_COL_CANDIDATES = [
    "Formatted_parameters",
    "Formatted Parameters",
    "Formatted_Parameters",
    "Experiment Parameters",
    "Parameters",
    "parameter",
    "params",
]

# For string-based matching fallback (if Experiment Index missing)
# We'll try these columns in the *trajectory file* as keys.
TRAJ_KEY_COL_CANDIDATES = [
    "Experiment Parameters",
    "Experiment parameter",
    "Parameters",
    "Selected Parameters",
]

# =========================
# Helpers
# =========================
def _norm(s: str) -> str:
    """Normalize whitespace for stable string matching."""
    return " ".join(str(s).strip().split())

def _pick_col(cols: List[str], candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in cols:
            return c
    return None

def _safe_read_csv(path: str) -> pd.DataFrame:
    return pd.read_csv(path)

def _ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def _dataset_out_dir(dataset_name: str) -> str:
    # Keep folder name same as dataset folder, but inside OUT_ROOT
    return os.path.join(OUT_ROOT, dataset_name)

def _list_llm_files(dataset_folder: str) -> List[str]:
    files = glob.glob(os.path.join(dataset_folder, LLM_GLOB))
    files = [f for f in files if not any(f.endswith(suf) for suf in EXCLUDE_SUFFIXES)]
    return sorted(set(files))

def _drop_last(df: pd.DataFrame) -> pd.DataFrame:
    if df.shape[0] <= 1:
        return df.copy()
    return df.iloc[:-1].copy()

def _extract_selected_indices_from_trajectory(df_traj: pd.DataFrame) -> Tuple[Optional[np.ndarray], str]:
    """
    Return (indices, method_string).
    Prefer Experiment Index. If missing, return (None, "no_index").
    """
    if "Experiment Index" in df_traj.columns:
        idx = pd.to_numeric(df_traj["Experiment Index"], errors="coerce").dropna().astype(int).to_numpy()
        return idx, "experiment_index"
    return None, "no_index"

def _subset_by_indices(df_orig: pd.DataFrame, idx: np.ndarray) -> Tuple[pd.DataFrame, Dict]:
    """
    Subset original df using integer positions.
    Returns subset df and stats.
    """
    stats = {}
    N = len(df_orig)

    idx = np.asarray(idx, dtype=int)
    stats["n_idx_raw"] = int(idx.size)
    stats["n_idx_unique"] = int(np.unique(idx).size)

    # valid range
    valid_mask = (idx >= 0) & (idx < N)
    idx_valid = idx[valid_mask]
    idx_invalid = idx[~valid_mask]

    stats["n_idx_valid"] = int(idx_valid.size)
    stats["n_idx_invalid"] = int(idx_invalid.size)
    stats["invalid_examples"] = idx_invalid[:10].tolist() if idx_invalid.size else []

    # Subset using iloc (keeps duplicates if present in idx_valid)
    df_sub = df_orig.iloc[idx_valid].copy()
    return df_sub, stats

def _subset_by_string_key(df_orig: pd.DataFrame, df_traj: pd.DataFrame) -> Tuple[pd.DataFrame, Dict, str]:
    """
    Fallback matching:
      - pick a key column from orig
      - pick a key column from traj
      - normalize strings
      - map traj key -> orig row (first occurrence)
    """
    stats = {}

    orig_key_col = _pick_col(list(df_orig.columns), ORIG_KEY_COL_CANDIDATES)
    traj_key_col = _pick_col(list(df_traj.columns), TRAJ_KEY_COL_CANDIDATES)

    if orig_key_col is None or traj_key_col is None:
        stats["error"] = f"Could not find key columns. orig_key_col={orig_key_col}, traj_key_col={traj_key_col}"
        return df_orig.iloc[0:0].copy(), stats, "string_key_missing_cols"

    # build lookup from orig key -> first row index
    orig_keys = df_orig[orig_key_col].astype(str).map(_norm)
    # dropna-ish + duplicates
    # keep first occurrence (consistent with your membrane mapping approach)
    orig_key_to_i = {}
    for i, k in enumerate(orig_keys):
        if k and k not in orig_key_to_i:
            orig_key_to_i[k] = i

    traj_keys = df_traj[traj_key_col].astype(str).map(_norm).to_numpy()
    idx = []
    missing = []
    for k in traj_keys:
        if k in orig_key_to_i:
            idx.append(orig_key_to_i[k])
        else:
            missing.append(k)

    stats["orig_key_col"] = orig_key_col
    stats["traj_key_col"] = traj_key_col
    stats["n_traj_keys"] = int(len(traj_keys))
    stats["n_matched"] = int(len(idx))
    stats["n_missing"] = int(len(missing))
    stats["missing_examples"] = missing[:5]

    df_sub = df_orig.iloc[idx].copy()
    return df_sub, stats, "string_key"

def _prepend_tracking_cols(df_sub: pd.DataFrame, df_traj_used: pd.DataFrame, traj_file: str) -> pd.DataFrame:
    """
    Add helpful tracking columns while keeping all original dataset columns.
    These extra columns are useful later and don't remove any original data.
    """
    out = df_sub.copy()

    # best-effort attach iteration if lengths align
    if "Iteration" in df_traj_used.columns and len(df_traj_used) == len(out):
        out.insert(0, "Iteration", df_traj_used["Iteration"].to_numpy())
    else:
        out.insert(0, "Iteration", np.arange(1, len(out) + 1))

    out.insert(0, "Trajectory_File", os.path.basename(traj_file))
    return out

def _save_subset(df_sub: pd.DataFrame, dataset_name: str, orig_csv_path: str, traj_file: str) -> str:
    out_dir = _dataset_out_dir(dataset_name)
    _ensure_dir(out_dir)

    orig_base = os.path.splitext(os.path.basename(orig_csv_path))[0]
    npts = df_sub.shape[0]
    traj_base = os.path.splitext(os.path.basename(traj_file))[0]

    # user asked: "original dataset name_number of data points"
    # to avoid overwrite across trajectories, append the trajectory basename too
    out_name = f"{orig_base}_{npts}__{traj_base}.csv"
    out_path = os.path.join(out_dir, out_name)
    df_sub.to_csv(out_path, index=False)
    return out_path

def process_one_dataset(dataset_name: str, orig_csv: str, y_col: str) -> None:
    dataset_folder = os.path.join(BASE_DIR, dataset_name)
    if not os.path.isdir(dataset_folder):
        print(f"\n[SKIP] Dataset folder not found: {dataset_folder}")
        return

    if not os.path.exists(orig_csv):
        print(f"\n[SKIP] Original dataset CSV not found: {orig_csv} (needed for {dataset_name})")
        return

    df_orig = _safe_read_csv(orig_csv)
    N = len(df_orig)

    llm_files = _list_llm_files(dataset_folder)
    print(f"\n==============================")
    print(f"Dataset: {dataset_name}")
    print(f"Original: {orig_csv}  (N={N})")
    if y_col in df_orig.columns:
        y = pd.to_numeric(df_orig[y_col], errors="coerce")
        print(f"Objective column '{y_col}': non-null={int(y.notna().sum())}, unique={int(y.dropna().nunique())}")
    else:
        print(f"Objective column '{y_col}': NOT FOUND in original dataset (ok for matching if using indices/keys)")
    print(f"Found LLM trajectory files: {len(llm_files)}")
    print(f"Output folder: {_dataset_out_dir(dataset_name)}")
    print(f"==============================\n")

    if not llm_files:
        return

    # Dataset-level sanity: duplicate objective values (useful if you later do value-based metrics)
    if y_col in df_orig.columns:
        y = pd.to_numeric(df_orig[y_col], errors="coerce").dropna()
        dup = int(len(y) - y.nunique())
        max_rep = int(y.value_counts().max()) if not y.empty else 0
        print(f"[SANITY] Duplicate target entries: {dup} (max repetition of a value: {max_rep})")

    # process each trajectory
    for traj_file in llm_files:
        df_traj = _safe_read_csv(traj_file)

        n_traj_raw = len(df_traj)
        df_traj_used = _drop_last(df_traj) if DROP_LAST_POINT else df_traj.copy()
        n_traj_used = len(df_traj_used)

        # 1) Prefer Experiment Index
        idx, idx_method = _extract_selected_indices_from_trajectory(df_traj_used)

        if idx is not None and idx.size:
            df_sub, st = _subset_by_indices(df_orig, idx)
            match_method = idx_method
            extra_stats = st
            # If we used indices, lengths should match "valid indices count"
            # df_sub keeps duplicates if idx has duplicates.
        else:
            # 2) Fallback: string key matching
            df_sub, st, match_method = _subset_by_string_key(df_orig, df_traj_used)
            extra_stats = st

        # attach tracking cols (keeps all original columns)
        df_sub_out = _prepend_tracking_cols(df_sub, df_traj_used.iloc[:len(df_sub)].copy(), traj_file)

        # save
        out_path = _save_subset(df_sub_out, dataset_name, orig_csv, traj_file)

        # ---- per-file sanity stats ----
        print(f"[FILE] {os.path.basename(traj_file)}")
        print(f"  Trajectory rows: raw={n_traj_raw}, used(after_drop_last={DROP_LAST_POINT})={n_traj_used}")
        print(f"  Match method: {match_method}")
        print(f"  Subset rows saved: {len(df_sub_out)} → {out_path}")

        # index stats if used
        if match_method == "experiment_index":
            print(f"  idx: raw={extra_stats.get('n_idx_raw')}, unique={extra_stats.get('n_idx_unique')}, "
                  f"valid={extra_stats.get('n_idx_valid')}, invalid={extra_stats.get('n_idx_invalid')}")
            if extra_stats.get("n_idx_invalid", 0) > 0:
                print(f"  invalid idx examples: {extra_stats.get('invalid_examples')}")

            # duplicate index warning
            if extra_stats.get("n_idx_unique", 0) < extra_stats.get("n_idx_raw", 0):
                print("  [WARN] Duplicate Experiment Index values found within this trajectory.")
        else:
            # string-match stats
            if "error" in extra_stats:
                print(f"  [WARN] {extra_stats['error']}")
            else:
                print(f"  key cols: traj='{extra_stats.get('traj_key_col')}', orig='{extra_stats.get('orig_key_col')}'")
                print(f"  matched={extra_stats.get('n_matched')}/{extra_stats.get('n_traj_keys')} "
                      f"(missing={extra_stats.get('n_missing')})")
                if extra_stats.get("n_missing", 0) > 0:
                    print(f"  missing examples: {extra_stats.get('missing_examples')}")

        # optional: objective distribution sanity for the subset
        if y_col in df_orig.columns and y_col in df_sub_out.columns:
            yy = pd.to_numeric(df_sub_out[y_col], errors="coerce").dropna()
            if not yy.empty:
                print(f"  subset objective '{y_col}': min={yy.min():.6g}, max={yy.max():.6g}, "
                      f"mean={yy.mean():.6g}, unique={yy.nunique()}")
        print("")

def main():
    _ensure_dir(OUT_ROOT)
    for ds, (orig_csv, y_col) in ORIGINAL_DATASETS.items():
        process_one_dataset(ds, orig_csv, y_col)

if __name__ == "__main__":
    main()

In [ ]:
import os
import re
import glob
import random
import hashlib
from pathlib import Path
from typing import Dict, Tuple, Sequence, List

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.utils import resample
from xgboost import XGBRegressor

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions.normal import Normal
from torch.utils.data import TensorDataset, DataLoader

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# Config
# ============================================================
DATASET_CSVS = {
    "Membrane_dataset": "Membrane_dataset.csv",
    "Perovskite_dataset": "Perovskite_dataset.csv",
    "P3HT_dataset": "P3HT_dataset.csv",
    "matbench_steels (composition)": "matbench_steels.csv",
}

SAMPLING_CHECK_ROOT = "Sampling Performance Check"
OUT_BASE = "al_trajectory_data_all"

MODEL_NAMES = ("BNN", "XGB", "RFR", "GPR")
ALPHAS = (0.1, 0.5, 1.0)

SUBSET_FILE_FILTER = re.compile(r".*llm_experiment_.*\.csv$", re.IGNORECASE)

# ============================================================
# Helpers
# ============================================================
_seed_re = re.compile(r"(?i)seed(\d+)")


def parse_seed_from_name(s: str, default: int = 42) -> int:
    m = _seed_re.search(s)
    return int(m.group(1)) if m else default


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def safe_name(s: str, max_len: int = 140) -> str:
    s = re.sub(r'[<>:"/\\|?*\n\r\t]', "_", str(s))
    s = re.sub(r"\s+", " ", s).strip()
    return s[:max_len]


def short_hash(s: str, n: int = 10) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:n]


def alpha_tag(alpha: float) -> str:
    a = float(alpha)
    if abs(a - round(a)) < 1e-12:
        return str(int(round(a)))
    s = f"{a:.6g}"
    return s.replace(".", "p").replace("-", "m")


# ============================================================
# Dataset utilities (YOUR exact setups)
# ============================================================
def is_minimization_dataset(dataset_name: str) -> bool:
    # Perovskite: you defined y_df = -Instability index (maximize -instability == minimize instability)
    return dataset_name == "Perovskite_dataset"


def get_objective_column(dataset_name: str) -> str:
    if dataset_name == "matbench_steels (composition)":
        return "yield strength"
    if dataset_name == "P3HT_dataset":
        return "Conductivity (measured) (S/cm)"
    if dataset_name == "Perovskite_dataset":
        return "Instability index"
    if dataset_name == "Membrane_dataset":
        return "Elastic Modulus_mean"
    raise ValueError(f"Unknown dataset_name: {dataset_name}")


def build_features_and_targets(df: pd.DataFrame, dataset_name: str) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """
    Returns:
      X_df (features in your defined column selection)
      y_true (original target, in original units/sign)
      y_eff  (effective target used for acquisition; for Perovskite it's negative)
    """
    if dataset_name == "matbench_steels (composition)":
        # Your snippet (note: you *finally* used iloc[:,2:16])
        X_df = df.iloc[:, 2:16].copy()
        y_true = pd.to_numeric(df["yield strength"], errors="coerce").to_numpy(dtype=float)
        y_eff = y_true.copy()
        return X_df, y_true, y_eff

    if dataset_name == "P3HT_dataset":
        X_df = df.iloc[:, 0:5].copy()
        y_true = pd.to_numeric(df["Conductivity (measured) (S/cm)"], errors="coerce").to_numpy(dtype=float)
        y_eff = y_true.copy()
        return X_df, y_true, y_eff

    if dataset_name == "Perovskite_dataset":
        X_df = df.iloc[:, 0:3].copy()
        y_true = pd.to_numeric(df["Instability index"], errors="coerce").to_numpy(dtype=float)
        y_eff = -y_true  # EXACTLY like your snippet
        return X_df, y_true, y_eff

    if dataset_name == "Membrane_dataset":
        X_df = df.drop([
            'Elastic Modulus_mean', 'Elastic Modulus_std', 'Yield Strength_mean',
            'Yield Strength_std', 'Creep Strain_mean', 'Creep Strain_std',
            'Plateau Slope_mean', 'Plateau Slope_std', 'Densification Slope_mean',
            'Densification Slope_std', 'Changepoint_mean', 'Changepoint_std', 'Date',
            'Average Standard Deviation_mean', 'Average Standard Deviation_std',
            'Coefficient of Variation_mean', 'Coefficient of Variation_std',
            'Coefficient of Variation', 'Batch', 'Sample', 'Index',
            'Formatted_Parameters', 'Report', 'Report with output',
            'Formatted_Parameters with output'
        ], axis=1, errors="ignore").copy()

        y_true = pd.to_numeric(df["Elastic Modulus_mean"], errors="coerce").to_numpy(dtype=float)
        y_eff = y_true.copy()
        return X_df, y_true, y_eff

    raise ValueError(f"Unknown dataset_name: {dataset_name}")


def coerce_numeric_X(X_df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure X is numeric; turn non-numeric into NaN so imputer can handle it.
    Also converts inf/-inf to NaN.
    """
    X_num = X_df.apply(pd.to_numeric, errors="coerce")
    X_num = X_num.replace([np.inf, -np.inf], np.nan)
    return X_num


# ============================================================
# Warm-start index recovery (robust)
# ============================================================
def recover_indices_from_subset(
    df_full: pd.DataFrame,
    df_subset: pd.DataFrame,
    *,
    index_col_candidates=("Experiment Index", "Experiment_Index", "Index", "index", "Idx", "ID", "id"),
) -> np.ndarray:
    for c in index_col_candidates:
        if c in df_subset.columns:
            idx = pd.to_numeric(df_subset[c], errors="coerce").dropna().astype(int).to_numpy()
            return idx

    ignore = {
        "Iteration", "Stopping Reason", "Phase",
        "Predicted Target Value", "Uncertainty",
        "Rerank_Score_v35", "Rerank_Score",
    }
    shared = [c for c in df_subset.columns if c in df_full.columns and c not in ignore]
    if not shared:
        raise ValueError("Cannot recover indices: no index column and no shared columns.")

    full_hash = pd.util.hash_pandas_object(df_full[shared], index=False)
    sub_hash = pd.util.hash_pandas_object(df_subset[shared], index=False)

    map_hash_to_indices: Dict[int, List[int]] = {}
    for i, h in enumerate(full_hash.values):
        map_hash_to_indices.setdefault(int(h), []).append(i)

    matched = []
    missing = 0
    for h in sub_hash.values:
        h = int(h)
        if h not in map_hash_to_indices or len(map_hash_to_indices[h]) == 0:
            missing += 1
            continue
        matched.append(map_hash_to_indices[h].pop(0))

    if missing > 0:
        raise ValueError(f"Subset row matching failed: missing {missing}/{len(df_subset)} rows.")
    return np.array(matched, dtype=int)


# ============================================================
# Models (Perovskite+GPR scaling fix kept)
# ============================================================
class GPR:
    def __init__(self, kernel=None, alpha: float = 1.0, random_state: int = 42, y_scale_factor: float = 1.0):
        self.alpha = float(alpha)
        self.y_scale_factor = float(y_scale_factor)
        self.kernel = kernel if kernel else (
            C(1.0, (1e-5, 1e5)) * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e3)) +
            WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-3, 1e6))
        )
        self.model = GaussianProcessRegressor(kernel=self.kernel, n_restarts_optimizer=10, random_state=random_state)

    def update_model(self, X_train: np.ndarray, y_train: np.ndarray):
        self.model.fit(X_train, y_train * self.y_scale_factor)

    def select_next_point(self, X_candidates: np.ndarray):
        mean_s, std_s = self.model.predict(X_candidates, return_std=True)
        mean = mean_s / self.y_scale_factor
        std = std_s / self.y_scale_factor
        ucb = mean + self.alpha * std
        return int(np.argmax(ucb)), ucb, mean, std


class RFR:
    def __init__(self, n_estimators=400, alpha: float = 1.0, random_state: int = 42):
        self.alpha = float(alpha)
        self.model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state)

    def update_model(self, X_train: np.ndarray, y_train: np.ndarray):
        self.model.fit(X_train, y_train)

    def rfPredictionIntervals(self, xVal: np.ndarray, percentile: float = 90):
        y_preds = np.stack([t.predict(xVal) for t in self.model.estimators_], axis=1)
        q_down = (100 - percentile) / 2.0
        q_up = 100 - q_down
        y_lower = np.percentile(y_preds, q_down, axis=1)
        y_upper = np.percentile(y_preds, q_up, axis=1)
        y_mean = self.model.predict(xVal)
        y_uncert = y_upper - y_lower
        return y_mean, y_uncert

    def select_next_point(self, X_candidates: np.ndarray):
        mean, uncertainty = self.rfPredictionIntervals(X_candidates)
        ucb = mean + self.alpha * uncertainty
        return int(np.argmax(ucb)), ucb, mean, uncertainty


class XGB:
    def __init__(self, n_estimators=400, n_models=30, alpha: float = 1.0, random_state: int = 42):
        self.alpha = float(alpha)
        self.n_models = int(n_models)
        self.n_estimators = int(n_estimators)
        self.random_state = int(random_state)
        self.models: List[XGBRegressor] = []

    def update_model(self, X_train: np.ndarray, y_train: np.ndarray):
        self.models = []
        rng = np.random.RandomState(self.random_state)
        for _ in range(self.n_models):
            X_s, y_s = resample(X_train, y_train, random_state=rng.randint(0, 10000))
            m = XGBRegressor(
                n_estimators=self.n_estimators,
                reg_alpha=0,
                scale_pos_weight=1,
                base_score=0.5,
                random_state=rng.randint(0, 10000),
            )
            m.fit(X_s, y_s)
            self.models.append(m)

    def select_next_point(self, X_candidates: np.ndarray):
        y_preds = np.column_stack([m.predict(X_candidates) for m in self.models])
        y_mean = y_preds.mean(axis=1)
        y_std = y_preds.std(axis=1)
        ucb = y_mean + self.alpha * y_std
        return int(np.argmax(ucb)), ucb, y_mean, y_std


# ---------------- BNN ----------------
class BayesianLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.weight_mu = nn.Parameter(torch.Tensor(out_features, in_features).uniform_(-0.2, 0.2))
        self.weight_log_sigma = nn.Parameter(torch.Tensor(out_features, in_features).fill_(-5.0))
        self.bias_mu = nn.Parameter(torch.Tensor(out_features).uniform_(-0.2, 0.2))
        self.bias_log_sigma = nn.Parameter(torch.Tensor(out_features).fill_(-5.0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w_sigma = torch.exp(self.weight_log_sigma)
        b_sigma = torch.exp(self.bias_log_sigma)
        w = self.weight_mu + w_sigma * torch.randn_like(self.weight_mu, device=x.device)
        b = self.bias_mu + b_sigma * torch.randn_like(self.bias_mu, device=x.device)
        return F.linear(x, w, b)

    def kl_loss(self) -> torch.Tensor:
        w_sigma = torch.exp(self.weight_log_sigma)
        b_sigma = torch.exp(self.bias_log_sigma)
        w_kl = (torch.log(1.0 / w_sigma) + (w_sigma**2 + self.weight_mu**2 - 1) / 2).sum()
        b_kl = (torch.log(1.0 / b_sigma) + (b_sigma**2 + self.bias_mu**2 - 1) / 2).sum()
        return w_kl + b_kl


class BayesianNN(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.layers = nn.ModuleList([
            BayesianLinear(input_dim, hidden_dim),
            BayesianLinear(hidden_dim, hidden_dim),
            BayesianLinear(hidden_dim, hidden_dim),
            BayesianLinear(hidden_dim, hidden_dim),
            BayesianLinear(hidden_dim, hidden_dim),
        ])
        self.out = BayesianLinear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = F.relu(layer(x))
        return self.out(x)

    def kl_loss(self) -> torch.Tensor:
        return sum(l.kl_loss() for l in self.layers) + self.out.kl_loss()


def elbo_loss(pred: torch.Tensor, y: torch.Tensor, kl: torch.Tensor, beta: float = 1.0) -> torch.Tensor:
    # pred: (B,1) or (B,) ; y: (B,) or (B,1)
    pred_vec = pred.view(-1)
    y_vec = y.view(-1)
    mse = F.mse_loss(pred_vec, y_vec, reduction="mean")
    return 0.5 * mse + beta * kl


def train_bnn(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    n_epochs: int = 600,
    lr: float = 1e-3,
    beta: float = 1.0,
    batch_size: int = 64,
    device: str = "cpu",
):
    ds = TensorDataset(X, y)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.to(device)
    for _ in range(n_epochs):
        for Xb, yb in dl:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            p = model(Xb)
            kl = model.kl_loss()
            loss = elbo_loss(p, yb, kl, beta)
            loss.backward()
            opt.step()


def predict_with_uncertainty(model: nn.Module, X: torch.Tensor, n_samples: int = 200):
    """
    Always return mean/std as 1D numpy arrays of length N (N = X.shape[0]).
    Works even when N==1.
    """
    model.eval()
    with torch.no_grad():
        preds = []
        for _ in range(n_samples):
            out = model(X)          # (N,1) typically
            out = out.view(-1)      # -> (N,)
            preds.append(out)
        preds = torch.stack(preds, dim=0)   # (S, N)

        mean = preds.mean(dim=0).cpu().numpy()  # (N,)
        std  = preds.std(dim=0).cpu().numpy()   # (N,)
    return mean, std


class BNN:
    def __init__(self, input_dim: int, device: str = "cpu", alpha: float = 1.0):
        self.alpha = float(alpha)
        self.device = device
        self.model = BayesianNN(input_dim).to(device)

    def update_model(self, X_train: np.ndarray, y_train: np.ndarray):
        Xt = torch.from_numpy(X_train).float().to(self.device)
        yt = torch.from_numpy(y_train).float().to(self.device)
        train_bnn(self.model, Xt, yt, device=self.device)

    def select_next_point(self, X_candidates: np.ndarray):
        Xc = torch.from_numpy(X_candidates).float().to(self.device)
        mean, std = predict_with_uncertainty(self.model, Xc)
        ucb = mean + self.alpha * std
        return int(np.argmax(ucb)), ucb, mean, std


def make_model(model_name: str, *, alpha: float, random_seed: int, input_dim: int, dataset_name: str):
    if model_name == "GPR":
        y_scale = 1e-3 if dataset_name == "Perovskite_dataset" else 1.0
        return GPR(alpha=alpha, random_state=random_seed, y_scale_factor=y_scale)
    if model_name == "RFR":
        return RFR(alpha=alpha, random_state=random_seed)
    if model_name == "XGB":
        return XGB(alpha=alpha, random_state=random_seed)
    if model_name == "BNN":
        device = "cuda" if torch.cuda.is_available() else "cpu"
        return BNN(input_dim, alpha=alpha, device=device)
    raise ValueError("Invalid model name")


# ============================================================
# AL runner (with imputation; min/max handled exactly as desired)
# ============================================================
def run_MLAL_warmstart(
    df_full: pd.DataFrame,
    dataset_name: str,
    model_name: str,
    alpha: float,
    random_seed: int,
    warmstart_indices: np.ndarray,
    *,
    run_kind: str,      # "warm" or "rand"
    subset_key: str,    # short identifier (already sanitized)
    out_dir: str,
) -> Tuple[str, pd.DataFrame]:
    set_all_seeds(random_seed)

    y_col = get_objective_column(dataset_name)

    # Build X/y using YOUR setups
    X_df_raw, y_true, y_eff = build_features_and_targets(df_full, dataset_name)
    X_df = coerce_numeric_X(X_df_raw)

    X = X_df.to_numpy(dtype=float)
    N = X.shape[0]
    d = X.shape[1]

    # Determine "best" in true space for logging / stopping
    minimize = is_minimization_dataset(dataset_name)
    if minimize:
        # y_eff = -y_true, maximizing y_eff => minimizing y_true
        best_true = float(np.nanmin(y_true))
        best_label = f"Min {y_col} in Dataset"
    else:
        best_true = float(np.nanmax(y_true))
        best_label = f"Max {y_col} in Dataset"

    warmstart_indices = np.asarray(warmstart_indices, dtype=int)
    warmstart_indices = warmstart_indices[(warmstart_indices >= 0) & (warmstart_indices < N)]
    warmstart_indices = np.unique(warmstart_indices)
    if warmstart_indices.size == 0:
        raise ValueError("No valid warm-start indices")

    # Initialize training
    selected = warmstart_indices.tolist()
    X_train = X[warmstart_indices]
    y_train_eff = y_eff[warmstart_indices]

    # ---- Fit imputer+scaler on TRAIN ONLY (warmstart), then apply to candidates
    imputer = SimpleImputer(strategy="mean")
    X_train_imp = imputer.fit_transform(X_train)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)

    model = make_model(model_name, alpha=alpha, random_seed=random_seed, input_dim=d, dataset_name=dataset_name)

    # Trajectory rows: log warmstart points as "observed"
    rows = []
    for t, idx in enumerate(warmstart_indices.tolist()):
        rows.append({
            "Iteration": int(t),
            "Index": int(idx),
            "Observed Target Value": float(y_true[idx]),
            "Predicted Target Value": np.nan,
            "Uncertainty": np.nan,
            best_label: float(best_true),
            "Phase": "WarmStart",
            "Stopping Reason": "WarmStart",
        })

    it = warmstart_indices.size
    while True:
        # Train surrogate
        model.update_model(X_train_scaled, y_train_eff)

        # Candidates
        available = np.array(sorted(set(range(N)) - set(selected)), dtype=int)
        if available.size == 0:
            break

        X_cand_imp = imputer.transform(X[available])
        X_cand_scaled = scaler.transform(X_cand_imp)

        pick_pos, ucb, mean_eff, unc_eff = model.select_next_point(X_cand_scaled)
        next_idx = int(available[pick_pos])
        selected.append(next_idx)

        # Update training set
        X_train = np.vstack([X_train, X[[next_idx]]])
        y_train_eff = np.append(y_train_eff, y_eff[next_idx])

        # Refit imputer+scaler on expanded training set (mirrors your original "refit scaler each iter")
        imputer = SimpleImputer(strategy="mean")
        X_train_imp = imputer.fit_transform(X_train)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_imp)

        # Logging: convert predictions to true-space
        pred_true = float(-mean_eff[pick_pos]) if minimize else float(mean_eff[pick_pos])
        unc_true = float(unc_eff[pick_pos])
        obs_true = float(y_true[next_idx])

        # Stopping: when global optimum reached in true-space
        stop = (obs_true <= best_true) if minimize else (obs_true >= best_true)

        rows.append({
            "Iteration": int(it),
            "Index": next_idx,
            "Observed Target Value": obs_true,
            "Predicted Target Value": pred_true,
            "Uncertainty": unc_true,
            best_label: float(best_true),
            "Phase": "ActiveLearning",
            "Stopping Reason": f"{best_label} reached" if stop else "Continuing",
        })

        it += 1
        if stop:
            break

    df_traj = pd.DataFrame(rows)

    # Save
    out_dir_path = Path(out_dir)
    out_dir_path.mkdir(parents=True, exist_ok=True)

    a_tag = alpha_tag(alpha)
    fname = f"{model_name}__{dataset_name}__a{a_tag}__seed{random_seed}__{run_kind}__{subset_key}.csv"
    fname = safe_name(fname, max_len=180)
    out_path = out_dir_path / fname

    df_traj.to_csv(out_path, index=False)
    return str(out_path), df_traj


# ============================================================
# Batch driver
# ============================================================
def run_all_warmstarts_for_dataset(
    dataset_name: str,
    original_csv_path: str,
    sampling_check_root: str = SAMPLING_CHECK_ROOT,
    out_base: str = OUT_BASE,
    model_names: Sequence[str] = MODEL_NAMES,
    alphas: Sequence[float] = ALPHAS,
):
    df_full = pd.read_csv(original_csv_path)
    y_col = get_objective_column(dataset_name)

    print("=" * 70)
    print(f"Dataset: {dataset_name}")
    print(f"Original: {original_csv_path} (N={len(df_full)})")
    print(f"Objective: {y_col} non-null={df_full[y_col].notna().sum()} unique={df_full[y_col].nunique(dropna=True)}")
    print("=" * 70)

    # Quick X sanity (using your feature config)
    X_df_raw, y_true, y_eff = build_features_and_targets(df_full, dataset_name)
    X_df_num = coerce_numeric_X(X_df_raw)
    X = X_df_num.to_numpy(dtype=float)
    print(f"[DATA CHECK] X shape={X.shape}  NaN cells={int(np.isnan(X).sum())}  NaN rows={int(np.isnan(X).any(axis=1).sum())}  NaN cols={int(np.isnan(X).any(axis=0).sum())}")

    subset_dir = os.path.join(sampling_check_root, dataset_name)
    all_csv = sorted(glob.glob(os.path.join(subset_dir, "*.csv")))
    subset_files = [p for p in all_csv if SUBSET_FILE_FILTER.match(os.path.basename(p))]

    if not subset_files:
        raise RuntimeError(f"No subset files found in {subset_dir} matching llm_experiment_*.csv")

    out_dir_warm = os.path.join(out_base, dataset_name, "ML_warmstart_from_LLMsubsets")
    out_dir_rand = os.path.join(out_base, dataset_name, "ML_random_matched_to_LLMsubsets")
    os.makedirs(out_dir_warm, exist_ok=True)
    os.makedirs(out_dir_rand, exist_ok=True)

    N = len(df_full)

    for subset_fp in subset_files:
        base = os.path.basename(subset_fp)
        seed = parse_seed_from_name(base, default=42)

        df_subset = pd.read_csv(subset_fp)
        warm_idx = recover_indices_from_subset(df_full, df_subset)
        warm_idx = np.unique(warm_idx)
        m = int(warm_idx.size)

        rng = np.random.RandomState(seed)
        rand_idx = rng.choice(np.arange(N), size=m, replace=False)

        # Short unique key for filenames
        human = safe_name(base.replace(".csv", ""), max_len=40)
        key = f"m{m}_seed{seed}_{short_hash(base, 10)}"
        subset_key = safe_name(f"{human[:25]}_{key}", max_len=70)

        print(f"\n[WARMSTART FILE] {base}")
        print(f"  seed={seed}  warmstart_points={m}  (N={N})")
        print(f"  subset_key={subset_key}")

        for model_name in model_names:
            for alpha in alphas:
                alpha = float(alpha)

                try:
                    p1, d1 = run_MLAL_warmstart(
                        df_full, dataset_name, model_name, alpha, seed, warm_idx,
                        run_kind="warm", subset_key=subset_key, out_dir=out_dir_warm
                    )
                except Exception as e:
                    print(f"  [FAIL] {model_name} alpha={alpha} warmstart: {e}")
                    continue

                try:
                    p2, d2 = run_MLAL_warmstart(
                        df_full, dataset_name, model_name, alpha, seed, rand_idx,
                        run_kind="rand", subset_key=subset_key, out_dir=out_dir_rand
                    )
                except Exception as e:
                    print(f"  [FAIL] {model_name} alpha={alpha} random: {e}")
                    continue

                al_steps_1 = int((d1["Phase"] == "ActiveLearning").sum())
                al_steps_2 = int((d2["Phase"] == "ActiveLearning").sum())

                if is_minimization_dataset(dataset_name):
                    best1 = float(np.nanmin(d1["Observed Target Value"]))
                    best2 = float(np.nanmin(d2["Observed Target Value"]))
                else:
                    best1 = float(np.nanmax(d1["Observed Target Value"]))
                    best2 = float(np.nanmax(d2["Observed Target Value"]))

                print(f"  [OK] {model_name} a={alpha}: warm(AL={al_steps_1}, best={best1:.6g}) | rand(AL={al_steps_2}, best={best2:.6g})")

    print("\nDone.")


# Run all datasets
for ds, csvp in DATASET_CSVS.items():
    run_all_warmstarts_for_dataset(ds, csvp)

# Sampling Performance Analysis - For Plotting

In [59]:
"""
Summarize ML warm-start AL trajectories:
- Count iterations needed to reach the best candidate
- EXCLUDE warm-start points from the iteration count
- Compare LLM warm-start vs random warm-start
Grouped by: dataset, model, alpha, #warmstart points, seed, subset_key

Assumptions about trajectory CSVs:
- Produced by your script with columns including:
  ["Phase", "Stopping Reason", "Observed Target Value", "Index", ...]
- Phase values: "WarmStart" and "ActiveLearning"
- Files live under:
  al_trajectory_data_all/<dataset>/ML_warmstart_from_LLMsubsets/*.csv
  al_trajectory_data_all/<dataset>/ML_random_matched_to_LLMsubsets/*.csv
"""

import os
import re
import glob
import numpy as np
import pandas as pd

# -----------------------
# CONFIG
# -----------------------
OUT_BASE = "al_trajectory_data_all"
WARM_DIRNAME = "ML_warmstart_from_LLMsubsets"
RAND_DIRNAME = "ML_random_matched_to_LLMsubsets"

# if you want to restrict to certain datasets:
DATASETS = None  # e.g. ["Membrane_dataset", "Perovskite_dataset"] or None for all found

# minimization datasets in YOUR setup (Perovskite uses y_df = -Instability index)
MINIMIZE_DATASETS = {"Perovskite_dataset"}  # add more if needed

# -----------------------
# Helpers
# -----------------------
def parse_fname_metadata(fname: str):
    """
    Expected filename pattern from your code:
      {model}__{dataset}__a{alphaTag}__seed{seed}__{run_kind}__{subset_key}.csv
    where alphaTag like 0p1 for 0.1, or 1 for 1.0.
    """
    base = os.path.basename(fname)
    stem = base[:-4] if base.lower().endswith(".csv") else base
    parts = stem.split("__")
    if len(parts) < 6:
        return None

    model = parts[0]
    dataset = parts[1]

    # alpha tag: e.g. "a0p1" or "a1"
    a_tag = parts[2]
    if not a_tag.startswith("a"):
        return None
    alpha_str = a_tag[1:].replace("p", ".").replace("m", "-")
    try:
        alpha = float(alpha_str)
    except Exception:
        alpha = np.nan

    # seed: e.g. "seed42"
    seed_part = parts[3]
    m = re.search(r"seed(\d+)", seed_part, re.IGNORECASE)
    seed = int(m.group(1)) if m else None

    run_kind = parts[4]  # "warm" or "rand"
    subset_key = "__".join(parts[5:])  # may include extra "__" if any

    return dict(
        model=model,
        dataset=dataset,
        alpha=alpha,
        seed=seed,
        run_kind=run_kind,
        subset_key=subset_key,
        file=fname,
    )

def warmstart_count_from_df(df: pd.DataFrame) -> int:
    if "Phase" not in df.columns:
        return 0
    return int((df["Phase"] == "WarmStart").sum())

def activelearning_rows(df: pd.DataFrame) -> pd.DataFrame:
    if "Phase" in df.columns:
        return df[df["Phase"] == "ActiveLearning"].copy()
    # fallback: assume all are AL
    return df.copy()

def iterations_to_best_excluding_warmstart(df: pd.DataFrame, dataset: str):
    """
    Returns:
      - iters_to_best: number of AL selections until best candidate is first reached (1-based),
                       excluding warmstart points.
      - reached_best: bool
      - best_value_true: best achieved within the run (max or min depending on dataset)
      - best_value_global: global best within this run's recorded data? (same as above)
    Notes:
      - We interpret "best candidate reached" as hitting the best *observed* value in the run
        (AL phase only) OR if "Stopping Reason" includes "reached".
      - If you want "dataset-global optimum", this code infers from the run itself.
        If you stored "Max ... in Dataset" / "Min ... in Dataset" column, we use that too.
    """
    df_al = activelearning_rows(df)
    if df_al.empty:
        return dict(iters_to_best=np.nan, reached_best=False, best_value_run=np.nan, best_value_dataset=np.nan)

    y = pd.to_numeric(df_al["Observed Target Value"], errors="coerce").to_numpy()

    minimize = dataset in MINIMIZE_DATASETS

    # Best value within AL trajectory (run)
    if minimize:
        best_run = np.nanmin(y)
    else:
        best_run = np.nanmax(y)

    # Find first AL step that hits the best_run (tolerant to float jitter)
    # Use exact match when possible, else isclose.
    hit_mask = np.isclose(y, best_run, rtol=0, atol=1e-12) | (y == best_run)
    if hit_mask.any():
        first_hit_idx0 = int(np.argmax(hit_mask))  # first True
        iters_to_best = first_hit_idx0 + 1  # 1-based count of AL selections
        reached_best = True
    else:
        iters_to_best = np.nan
        reached_best = False

    # If your file includes dataset-best column, try to read it
    best_dataset = np.nan
    # columns like "Max Instability index in Dataset" or "Min Instability index in Dataset"
    for c in df.columns:
        if (" in Dataset" in c) and (("Max " in c) or ("Min " in c)):
            # take first non-null
            v = pd.to_numeric(df[c], errors="coerce").dropna()
            if len(v):
                best_dataset = float(v.iloc[0])
                break

    return dict(
        iters_to_best=iters_to_best,
        reached_best=reached_best,
        best_value_run=float(best_run),
        best_value_dataset=float(best_dataset) if not np.isnan(best_dataset) else np.nan,
    )

# -----------------------
# Load all warm/rand trajectory files
# -----------------------
warm_files = glob.glob(os.path.join(OUT_BASE, "*", WARM_DIRNAME, "*.csv"))
rand_files = glob.glob(os.path.join(OUT_BASE, "*", RAND_DIRNAME, "*.csv"))

all_files = warm_files + rand_files
meta_rows = []
for fp in all_files:
    m = parse_fname_metadata(fp)
    if m is None:
        continue
    if DATASETS is not None and m["dataset"] not in DATASETS:
        continue
    meta_rows.append(m)

if not meta_rows:
    raise RuntimeError("No trajectory files found (or filename parsing failed).")

meta_df = pd.DataFrame(meta_rows)

# -----------------------
# Compute summary per file
# -----------------------
summ_rows = []
for _, r in meta_df.iterrows():
    fp = r["file"]
    df = pd.read_csv(fp)

    m = warmstart_count_from_df(df)

    stats = iterations_to_best_excluding_warmstart(df, r["dataset"])
    summ_rows.append({
        "dataset": r["dataset"],
        "model": r["model"],
        "alpha": r["alpha"],
        "seed": r["seed"],
        "run_kind": r["run_kind"],  # warm or rand
        "subset_key": r["subset_key"],
        "warmstart_points": m,
        "iters_to_best_excl_warmstart": stats["iters_to_best"],
        "reached_best": stats["reached_best"],
        "best_value_run": stats["best_value_run"],
        "best_value_dataset_col": stats["best_value_dataset"],
        "file": fp,
    })

summary_df = pd.DataFrame(summ_rows)

# -----------------------
# Pair warm vs rand for same dataset/model/alpha/seed/subset_key/warmstart_points
# -----------------------
key_cols = ["dataset", "model", "alpha", "seed", "subset_key", "warmstart_points"]
warm_df = summary_df[summary_df["run_kind"] == "warm"].copy()
rand_df = summary_df[summary_df["run_kind"] == "rand"].copy()

paired = warm_df.merge(
    rand_df,
    on=key_cols,
    how="inner",
    suffixes=("_warm", "_rand"),
)

# -----------------------
# Aggregate: mean/median iterations to best over repeats (seeds/subsets)
# Group by dataset, model, alpha, warmstart_points
# -----------------------
group_cols = ["dataset", "model", "alpha", "warmstart_points"]

def agg_block(df, col):
    return pd.Series({
        "n": df[col].notna().sum(),
        "mean": float(np.nanmean(df[col])) if df[col].notna().any() else np.nan,
        "median": float(np.nanmedian(df[col])) if df[col].notna().any() else np.nan,
        "std": float(np.nanstd(df[col])) if df[col].notna().any() else np.nan,
    })

agg_rows = []
for g, sub in paired.groupby(group_cols):
    dset, model, alpha, m = g
    w = agg_block(sub, "iters_to_best_excl_warmstart_warm")
    r = agg_block(sub, "iters_to_best_excl_warmstart_rand")
    agg_rows.append({
        "dataset": dset,
        "model": model,
        "alpha": alpha,
        "warmstart_points": m,

        "warm_n": w["n"],
        "warm_mean_iters": w["mean"],
        "warm_median_iters": w["median"],
        "warm_std_iters": w["std"],

        "rand_n": r["n"],
        "rand_mean_iters": r["mean"],
        "rand_median_iters": r["median"],
        "rand_std_iters": r["std"],

        "delta_mean_warm_minus_rand": w["mean"] - r["mean"] if (not np.isnan(w["mean"]) and not np.isnan(r["mean"])) else np.nan,
        "delta_median_warm_minus_rand": w["median"] - r["median"] if (not np.isnan(w["median"]) and not np.isnan(r["median"])) else np.nan,
    })

agg_df = pd.DataFrame(agg_rows).sort_values(["dataset", "model", "alpha", "warmstart_points"])

# -----------------------
# Save outputs
# -----------------------
out_summary = os.path.join(OUT_BASE, "warmstart_vs_random__per_file_summary.csv")
out_paired  = os.path.join(OUT_BASE, "warmstart_vs_random__paired_runs.csv")
out_agg     = os.path.join(OUT_BASE, "warmstart_vs_random__aggregated.csv")

summary_df.to_csv(out_summary, index=False)
paired.to_csv(out_paired, index=False)
agg_df.to_csv(out_agg, index=False)

print(f"[OK] Per-file summary -> {out_summary}")
print(f"[OK] Paired warm vs rand -> {out_paired}")
print(f"[OK] Aggregated summary -> {out_agg}")

# Show a quick table in notebook
display_cols = [
    "dataset", "model", "alpha", "warmstart_points",
    "warm_mean_iters", "rand_mean_iters",
    "delta_mean_warm_minus_rand",
    "warm_median_iters", "rand_median_iters",
    "delta_median_warm_minus_rand",
    "warm_n", "rand_n"
]
print("\n=== Aggregated (mean/median iters to best, excluding warmstart) ===")
print(agg_df[display_cols].to_string(index=False))

[OK] Per-file summary -> al_trajectory_data_all\warmstart_vs_random__per_file_summary.csv
[OK] Paired warm vs rand -> al_trajectory_data_all\warmstart_vs_random__paired_runs.csv
[OK] Aggregated summary -> al_trajectory_data_all\warmstart_vs_random__aggregated.csv

=== Aggregated (mean/median iters to best, excluding warmstart) ===
                      dataset model  alpha  warmstart_points  warm_mean_iters  rand_mean_iters  delta_mean_warm_minus_rand  warm_median_iters  rand_median_iters  delta_median_warm_minus_rand  warm_n  rand_n
             Membrane_dataset   BNN    0.1                 5        38.000000        26.000000                   12.000000               38.0               26.0                          12.0     1.0     1.0
             Membrane_dataset   BNN    0.1                 6        19.500000        13.000000                    6.500000               19.5               13.0                           6.5     2.0     2.0
             Membrane_dataset   BNN    0.1    